# Creating features

The columns of a table are what someone recorded, not necessarily what a model needs. A feature computed from the others can state directly what a model would otherwise have to discover by itself, or could never discover: rooms per household instead of the total rooms and the total households of a district, the age of a car instead of the date it was registered. Creating such features, usually from knowledge of the domain, is called **feature engineering**.

## Ratios

The California districts of [Outliers and capped values](outliers.ipynb). New features are tried on the training rows only, since choosing them is a decision about the model:

In [ ]:
import tarfile
import urllib.request
from pathlib import Path

import pandas as pd
from sklearn.model_selection import train_test_split

housing_csv = Path("data/housing/housing.csv")
if not housing_csv.exists():
    housing_csv.parent.mkdir(parents=True, exist_ok=True)
    archive, _ = urllib.request.urlretrieve("https://github.com/ageron/data/raw/main/housing.tgz")
    with tarfile.open(archive) as tar:
        tar.extractall(path="data", filter="data")
housing = pd.read_csv(housing_csv)
train, test = train_test_split(housing, test_size=0.2, random_state=42)
train.head(3)

,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,median_house_value,ocean_proximity
14196,-117.22,32.75,34.0,6001.0,1111.0,2654.0,1072.0,4.5878,291000.0,NEAR OCEAN
8267,-117.03,32.69,10.0,901.0,163.0,698.0,167.0,4.6648,156100.0,NEAR OCEAN
17445,-122.27,37.74,28.0,6909.0,1554.0,2974.0,1484.0,3.6875,353900.0,NEAR BAY


The totals (rooms, bedrooms, people, households) mostly measure how big a district is, which says little about its homes: a district of 3,000 rooms may be 1,000 small flats or 300 large houses. Ratios describe the typical home:

In [ ]:
import numpy as np
def add_ratios(frame):
    return frame.assign(
        rooms_per_household=frame["total_rooms"] / frame["households"],
        bedrooms_ratio=frame["total_bedrooms"] / frame["total_rooms"],
        people_per_household=frame["population"] / frame["households"],
    )


with_ratios = add_ratios(train)
with_ratios.corr(numeric_only=True)["median_house_value"].drop("median_house_value").sort_values(
    key=np.abs, ascending=False).round(3)

median_income           0.687
bedrooms_ratio         -0.255
rooms_per_household     0.147
latitude               -0.146
total_rooms             0.134
housing_median_age      0.105
households              0.066
total_bedrooms          0.050
longitude              -0.044
population             -0.026
people_per_household   -0.022
Name: median_house_value, dtype: float64

The share of rooms that are bedrooms correlates with the value more than any total does (−0.26: districts where fewer rooms are bedrooms, so homes with more living space, are worth more), and so do the rooms per household (0.15). People per household has a correlation near 0, but [Skewed distributions](skewed_distributions.ipynb) showed a real relationship hidden by its extreme values: a correlation is a first look, not a verdict ([Correlation and what it misses](correlation.ipynb)).

The real test is a model. The ratios are computed inside the pipeline by a `FunctionTransformer`, which applies a function to whatever passes through it, here the `add_ratios` above. A linear regression and a random forest, with the original numerical features and with the three ratios added:

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import FunctionTransformer, StandardScaler

numerical = ["longitude", "latitude", "housing_median_age", "total_rooms", "total_bedrooms", "population",
             "households", "median_income"]
models = {
    "linear regression": LinearRegression(),
    "random forest": RandomForestRegressor(n_estimators=50, random_state=0, n_jobs=-1),
}
preprocessing = {
    "original features": [SimpleImputer(strategy="median"), StandardScaler()],
    "with the ratios": [FunctionTransformer(add_ratios), SimpleImputer(strategy="median"), StandardScaler()],
}
scores = pd.DataFrame({
    (features_name, model_name): cross_val_score(
        make_pipeline(*steps, model), train[numerical], train["median_house_value"], cv=5)
    for features_name, steps in preprocessing.items()
    for model_name, model in models.items()
})
pd.DataFrame({"mean R²": scores.mean(), "standard deviation across folds": scores.std()}).round(3)

mean R²  standard deviation across folds
original features linear regression    0.632                            0.014
                  random forest        0.811                            0.003
with the ratios   linear regression    0.641                            0.010
                  random forest        0.800                            0.007

Both versions are scored on the same five folds, so the change within each fold is a fairer comparison than the spread between folds, which mostly reflects how hard each fold is:

In [ ]:
change = scores["with the ratios"] - scores["original features"]
pd.DataFrame({"mean change in R²": change.mean(), "folds where the ratios helped": (change > 0).sum()}).round(3)

,mean change in R²,folds where the ratios helped
linear regression,0.008,4
random forest,-0.011,0


The linear regression gains a little under a point of R², and in four of the five folds: it can only add up weighted features, so a ratio is something it couldn't build by itself. The random forest doesn't gain; it loses about a point, in every fold. Trees combine features through successive splits, so with location and income they can approximate much of what the ratios add; why the extra columns then cost the forest a little is something this table doesn't explain. A feature that helps one model need not help another, so features are kept or dropped by measuring, as here.

## Dates

A date is not usable as it is, but it contains numbers that are. Four cars sold, with the dates of their registration and sale:

In [ ]:
sales = pd.DataFrame({
    "registered": pd.to_datetime(["2016-03-14", "2019-07-01", "2012-11-23", "2021-02-09"]),
    "sold": pd.to_datetime(["2024-12-20", "2025-01-05", "2025-06-15", "2025-07-01"]),
})
sales.assign(
    age_years=((sales["sold"] - sales["registered"]).dt.days / 365.25).round(1),
    sale_year=sales["sold"].dt.year,
    sale_month=sales["sold"].dt.month,
    sale_weekday=sales["sold"].dt.day_name(),
)

,registered,sold,age_years,sale_year,sale_month,sale_weekday
0,2016-03-14,2024-12-20,8.8,2024,12,Friday
1,2019-07-01,2025-01-05,5.5,2025,1,Sunday
2,2012-11-23,2025-06-15,12.6,2025,6,Sunday
3,2021-02-09,2025-07-01,4.4,2025,7,Tuesday


The **time elapsed** between two dates (the car's age when sold) is usually the most informative; the year, the month and the day of the week capture trends and seasons.

The month has a trap. As a number, December is 12 and January 1, eleven apart, while they are as close as June and July. A **cyclic encoding** places the months around a circle, with a sine and a cosine, so that the distance between two months is the same wherever the year turns:

In [ ]:
month = sales["sold"].dt.month
cyclic = pd.DataFrame({"month_sin": np.sin(2 * np.pi * month / 12), "month_cos": np.cos(2 * np.pi * month / 12)})


def gap(table, first, second):
    return np.sqrt(((table.iloc[first] - table.iloc[second]) ** 2).sum())


pd.DataFrame({
    "month as a number": [gap(month.to_frame(), 0, 1), gap(month.to_frame(), 2, 3)],
    "month on a circle": [gap(cyclic, 0, 1), gap(cyclic, 2, 3)],
}, index=["December to January", "June to July"]).round(2)

,month as a number,month on a circle
December to January,11.0,0.52
June to July,1.0,0.52


On the circle, both pairs of consecutive months are 0.52 apart. The same applies to hours of the day and days of the week. One-hot encoding the month is the other common choice ([Encoding categories](../categorical_encoding.ipynb)): twelve columns with no order at all, rather than two with the circle's order.

## A feature inside a text

[Kinds of features](kinds_of_features.ipynb) found that the Titanic passengers' names are useless as they are, almost one per passenger, but contain a title. A regular expression takes the word between the comma and the full stop:

In [ ]:
from sklearn.datasets import fetch_openml

passengers = fetch_openml("titanic", version=1, as_frame=True, data_home="data").frame
passengers_train, passengers_test = train_test_split(passengers, test_size=0.2, stratify=passengers["survived"], random_state=42)
titles = passengers_train["name"].str.extract(r",\s*([^.]+)\.")[0].rename("title")
titles.value_counts()

title
Mr              606
Miss            210
Mrs             156
Master           51
Rev               6
Dr                6
Col               3
Mlle              2
Ms                2
Major             1
Don               1
the Countess      1
Capt              1
Dona              1
Name: count, dtype: int64

Four common titles and ten rare ones. Grouping them by what they mean is a fixed rule, decided from knowledge of the period, not from the data: *Mlle* and *Ms* as *Miss*, *Mme* as *Mrs*, and the rest (clergy, doctors, officers, nobility) as *rare*. The survival rate of each group:

In [ ]:
title_groups = {"Mr": "Mr", "Mrs": "Mrs", "Mme": "Mrs", "Miss": "Miss", "Mlle": "Miss", "Ms": "Miss", "Master": "Master"}
title = titles.map(title_groups).fillna("rare").rename("title")
passengers_train["survived"].astype(int).groupby(title).agg(["mean", "size"]).round(2)

,mean,size
title,,
Master,0.49,51
Miss,0.67,214
Mr,0.17,606
Mrs,0.79,156
rare,0.30,20


*Master* was the title of boys: 49% of them survived, against 17% of the men called *Mr*. The title separates boys from men, which the sex can't, and it does so for every passenger, while the age, which could do the same, is missing for one in five ([Missing values](../missing_values.ipynb)).

The cabin, missing for most passengers, has a feature inside it too: its first letter is the deck:

In [ ]:
deck = passengers_train["cabin"].str[0].rename("deck")
passengers_train["survived"].astype(int).groupby(deck, dropna=False).agg(["mean", "size"]).round(2)

,mean,size
deck,,
A,0.50,16
B,0.69,48
C,0.62,73
D,0.68,41
E,0.74,31
F,0.67,12
G,0.67,3
T,0.00,1
NaN,0.31,822


On the decks with more than a handful of passengers, between 50% and 74% survived, against 31% of the passengers without a recorded cabin (`NaN`). The deck adds a little to what was already known from whether the cabin was recorded, and only for the fifth of the passengers who have one.

## Bins and products

Two transformers create features that let a linear model fit shapes other than a straight line:

- **Bins.** `KBinsDiscretizer` cuts a number into intervals and encodes them as categories, so that a linear model can give each interval its own weight: a step function instead of a line. With `strategy="quantile"`, the edges are chosen so that each interval holds about the same number of training rows; they are learned:

In [ ]:
from sklearn.preprocessing import KBinsDiscretizer

incomes = pd.DataFrame({"income": [18, 22, 25, 28, 31, 35, 60, 240]})
binner = KBinsDiscretizer(n_bins=3, encode="ordinal", strategy="quantile", quantile_method="averaged_inverted_cdf")
binner.fit(incomes)
binner.bin_edges_[0]

array([ 18.,  25.,  35., 240.])

- **Products.** `PolynomialFeatures` adds the squares of the features and their products two by two (with `degree=2`). A product of two features, an *interaction*, lets the effect of one depend on the other; a square lets a line become a curve. The parabola of [Correlation and what it misses](correlation.ipynb), which a straight line can't follow at all:

In [ ]:
from sklearn.preprocessing import PolynomialFeatures

parabola = pd.DataFrame({"x": np.linspace(-3, 3, 61)})
y_parabola = parabola["x"] ** 2
pd.Series({
    "x": LinearRegression().fit(parabola, y_parabola).score(parabola, y_parabola),
    "x and x²": make_pipeline(PolynomialFeatures(degree=2), LinearRegression()).fit(parabola, y_parabola).score(parabola, y_parabola),
}).round(3).rename("R² of a linear regression")

x           0.0
x and x²    1.0
Name: R² of a linear regression, dtype: float64

With $x^2$ as a feature, the linear regression fits the parabola exactly. The number of products grows quickly with the number of features (55 for ten features at degree 2), and so does the risk of overfitting, which is why polynomial features usually go with a regularised model ([Regularisation](../../supervised-learning/ml-fundamentals/regularization.ipynb)).

## Computed from each row, or learned

Every feature in this notebook belongs to one of two kinds, and the difference decides where it is computed:

| Kind | Examples | Where |
| --- | --- | --- |
| Computed from each row alone, by a fixed rule | Ratios, the time elapsed between two dates, the month as a sine and a cosine, a title extracted from a name and grouped by meaning, a product of two columns | Anywhere: once, when the table is built, for everyone who uses it, or in the model's pipeline |
| Learned from the data | Bins with edges taken from the data, the average target of each category ([Encoding categories](../categorical_encoding.ipynb)), the similarity to clusters of locations (Unsupervised learning *(coming)*), anything computed from other rows | Only in the model's pipeline, fitted on the training rows |

A fixed-rule feature is often put in the pipeline anyway, as `add_ratios` was here, so that the pipeline can take a raw row in production and compute everything itself ([California housing with pipelines](../../supervised-learning/basic-workflows/california_with_pipelines.ipynb)). A learned feature has no choice: computed from all the rows, it lets the test rows shape the training data. [Where data preparation ends](../where_preparation_ends.ipynb) draws that line for every step of this topic.